# DLS final project: Yandex Maps relevance

Ноутбук для Colab/GPU. Задача: предсказывать релевантность пары `запрос - организация`.

Классы:

- `0.0` - нерелевантно
- `0.1` - частично релевантно
- `1.0` - релевантно

Основная метрика из задания - `accuracy`.

Важно: `relevance_new` в eval не используем для подбора параметров, промптов или анализа ошибок. Для настройки используем только train/valid split из `data_for_train.jsonl`

## 0. Подготовка

1. Включи GPU: `Runtime -> Change runtime type -> T4 GPU` или лучше.
2. Надёжный вариант: загрузи готовый `yandex_maps_data.zip` в `/content/`. Ноутбук проверит SHA-256 и сам распакует его в `/content/data/`.
3. Альтернативно можно загрузить отдельные файлы:
   - `/content/data/data_for_train.jsonl`
   - `/content/data/data_for_eval.jsonl`
4. Запускай ячейки сверху вниз.

Если файлы лежат на Google Drive, можно поменять `DATA_DIR` в конфиге ниже.

In [ ]:
!pip -q install -U "transformers>=4.44" "datasets>=2.20" "accelerate>=0.33" "evaluate>=0.4" "sentencepiece"

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import random
import zipfile
import warnings

import numpy as np
import pandas as pd
import torch

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_DIR = Path("/content/data")
DATA_ZIP_PATH = Path("/content/yandex_maps_data.zip")
EXPECTED_DATA_ZIP_SHA256 = "19aec81812909d2616fa9f5aca448003a833f4a99b437242710f5b2d05a892e1"

if DATA_ZIP_PATH.exists():
    actual_sha256 = hashlib.sha256(DATA_ZIP_PATH.read_bytes()).hexdigest()
    if actual_sha256 != EXPECTED_DATA_ZIP_SHA256:
        raise ValueError(
            f"Поврежден ZIP: SHA256={actual_sha256}, ожидается {EXPECTED_DATA_ZIP_SHA256}. "
            "Загрузите yandex_maps_data.zip заново."
        )
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATA_ZIP_PATH) as archive:
        archive.extractall(DATA_DIR)
    print("Extracted verified data archive to", DATA_DIR)

TRAIN_PATH = DATA_DIR / "data_for_train.jsonl"
EVAL_PATH = DATA_DIR / "data_for_eval.jsonl"
OUTPUT_DIR = Path("/content/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALID_SIZE = 0.2

LABEL_VALUES = [0.0, 0.1, 1.0]
LABEL_TO_ID = {0.0: 0, 0.1: 1, 1.0: 2}
ID_TO_LABEL = {v: k for k, v in LABEL_TO_ID.items()}
LABEL_IDS = [LABEL_TO_ID[x] for x in LABEL_VALUES]
LABEL_NAMES = [str(x) for x in LABEL_VALUES]

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Train path:", TRAIN_PATH)
print("Eval path:", EVAL_PATH)

## 1. Загрузка данных и общий препроцессинг

Функция `build_model_text` собирает один текст из запроса и всех полей организации. Один и тот же текст потом используется для TF-IDF и трансформера.

In [ ]:
def load_jsonl(path: Path, skip_bad_tail: bool = True) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Не найден файл: {path}. Загрузите данные в /content/data/ или поменяйте DATA_DIR."
        )
    rows = []
    pending_variants = []
    pending_start_line = None
    last_error = None
    recovered_multiline = 0

    def try_parse(text: str):
        return json.loads(text)

    with path.open("r", encoding="utf-8-sig", errors="replace") as f:
        for line_no, line in enumerate(f, 1):
            line = line.rstrip()
            if not line:
                continue

            if pending_variants:
                next_variants = []
                for pending in pending_variants:
                    # Try common transfer/copy corruptions:
                    # 1) escaped "\n" became a physical line break inside a JSON string
                    # 2) object was split on regular JSON whitespace
                    # 3) object was wrapped without a separator
                    for sep in ("\\n", " ", ""):
                        candidate = pending + sep + line
                        try:
                            rows.append(try_parse(candidate))
                            pending_variants = []
                            pending_start_line = None
                            recovered_multiline += 1
                            break
                        except json.JSONDecodeError as exc:
                            last_error = exc
                            next_variants.append(candidate)
                    if not pending_variants:
                        break
                if pending_variants:
                    pending_variants = list(dict.fromkeys(next_variants))[:30]
                    too_long = line_no - pending_start_line > 200
                    too_large = max(len(item) for item in pending_variants) > 2_000_000
                    if too_long or too_large:
                        preview = pending_variants[0][:500]
                        raise ValueError(
                            f"Cannot recover JSON object in {path}, starting at line "
                            f"{pending_start_line}. Last error: {last_error}. "
                            f"Preview: {preview!r}. Перезагрузите файл: он, вероятно, поврежден."
                        ) from last_error
                continue

            try:
                rows.append(try_parse(line))
            except json.JSONDecodeError as exc:
                pending_variants = [line]
                pending_start_line = line_no
                last_error = exc

    if pending_variants:
        preview = pending_variants[0][:500]
        message = (
            f"Unfinished JSON object in {path}, starting at line {pending_start_line}. "
            f"Last error: {last_error}. Preview: {preview!r}. "
            f"Перезагрузите файл: он, вероятно, обрезан или поврежден."
        )
        if skip_bad_tail:
            print("WARNING:", message)
            print("Skipped the unfinished tail record and kept", len(rows), "complete rows.")
        else:
            raise ValueError(message) from last_error

    if recovered_multiline:
        print(f"Recovered {recovered_multiline} multiline JSON records from {path}.")
    return pd.DataFrame(rows)


def clean_value(value) -> str:
    if value is None or pd.isna(value):
        return ""
    return str(value).strip()


def build_model_text(row: pd.Series) -> str:
    query = clean_value(row.get("Text"))
    name = clean_value(row.get("name"))
    rubric = clean_value(row.get("normalized_main_rubric_name_ru"))
    address = clean_value(row.get("address"))
    prices = clean_value(row.get("prices_summarized"))
    reviews = clean_value(row.get("reviews_summarized"))

    return (
        f"Запрос пользователя: {query}\n"
        f"Название организации: {name}\n"
        f"Рубрика: {rubric}\n"
        f"Адрес: {address}\n"
        f"Товары и услуги: {prices}\n"
        f"Отзывы и описание: {reviews}"
    )


def encode_labels(labels: pd.Series) -> pd.Series:
    encoded = labels.map(LABEL_TO_ID)
    if encoded.isna().any():
        unknown = sorted(labels.loc[encoded.isna()].unique())
        raise ValueError(f"Unknown labels: {unknown}")
    return encoded.astype(int)


def decode_label_ids(label_ids) -> list:
    return [ID_TO_LABEL[int(label_id)] for label_id in label_ids]


train_data = load_jsonl(TRAIN_PATH)
eval_data = load_jsonl(EVAL_PATH)

EXPECTED_TRAIN_ROWS = 34_094
EXPECTED_EVAL_ROWS = 570

if len(train_data) != EXPECTED_TRAIN_ROWS or len(eval_data) != EXPECTED_EVAL_ROWS:
    raise ValueError(
        "Загружены неполные или поврежденные данные. "
        f"Получено train={len(train_data)}, eval={len(eval_data)}; "
        f"ожидается train={EXPECTED_TRAIN_ROWS}, eval={EXPECTED_EVAL_ROWS}. "
        "Перезагрузите исходные JSONL-файлы перед обучением."
    )

train_data["text"] = train_data.apply(build_model_text, axis=1)
train_data["label_id"] = encode_labels(train_data["relevance"])
eval_data["text"] = eval_data.apply(build_model_text, axis=1)

print("Train shape:", train_data.shape)
print("Eval shape:", eval_data.shape)
display(train_data.head(3))
display(train_data["relevance"].value_counts().sort_index())

In [ ]:
train_df, valid_df = train_test_split(
    train_data,
    test_size=VALID_SIZE,
    random_state=SEED,
    stratify=train_data["label_id"],
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

print("Train split:", train_df.shape)
print("Valid split:", valid_df.shape)
display(valid_df["relevance"].value_counts().sort_index())

valid_df[["Text", "name", "normalized_main_rubric_name_ru", "relevance"]].to_csv(
    OUTPUT_DIR / "valid_split_preview.csv",
    index=False,
    encoding="utf-8-sig",
)

## 2. Быстрый TF-IDF baseline

Это CPU-бейзлайн и контрольная точка. На локальном запуске лучшая конфигурация дала около `0.5969` accuracy на таком же split: TF-IDF word/char + `RidgeClassifier`.

In [ ]:
def make_tfidf_pipeline() -> Pipeline:
    word_vectorizer = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        max_features=160_000,
        sublinear_tf=True,
        token_pattern=r"(?u)\b\w+\b",
    )
    char_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_features=160_000,
        sublinear_tf=True,
    )
    return Pipeline(
        [
            ("features", FeatureUnion([("word", word_vectorizer), ("char", char_vectorizer)])),
            ("classifier", RidgeClassifier(alpha=1.0, random_state=SEED)),
        ]
    )


tfidf_model = make_tfidf_pipeline()
tfidf_model.fit(train_df["text"], train_df["label_id"])
tfidf_valid_pred_ids = tfidf_model.predict(valid_df["text"])
tfidf_valid_pred_labels = decode_label_ids(tfidf_valid_pred_ids)

tfidf_accuracy = accuracy_score(valid_df["label_id"], tfidf_valid_pred_ids)
print(f"TF-IDF validation accuracy: {tfidf_accuracy:.4f}")
print(
    classification_report(
        valid_df["label_id"],
        tfidf_valid_pred_ids,
        labels=LABEL_IDS,
        target_names=LABEL_NAMES,
        digits=4,
        zero_division=0,
    )
)

tfidf_cm = pd.DataFrame(
    confusion_matrix(valid_df["label_id"], tfidf_valid_pred_ids, labels=LABEL_IDS),
    index=[f"true_{x}" for x in LABEL_VALUES],
    columns=[f"pred_{x}" for x in LABEL_VALUES],
)
display(tfidf_cm)

tfidf_valid_out = valid_df.copy()
tfidf_valid_out["predicted_relevance"] = tfidf_valid_pred_labels
tfidf_valid_out["is_correct"] = (
    tfidf_valid_out["relevance"] == tfidf_valid_out["predicted_relevance"]
)
tfidf_valid_out.to_csv(
    OUTPUT_DIR / "tfidf_valid_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)
tfidf_valid_out.loc[~tfidf_valid_out["is_correct"]].head(300).to_csv(
    OUTPUT_DIR / "tfidf_errors_sample.csv",
    index=False,
    encoding="utf-8-sig",
)

## 3. GPU transformer baseline

По умолчанию стоит лёгкая модель `cointegrated/rubert-tiny2`, чтобы быстро проверить пайплайн. Для финального эксперимента на GPU можно заменить на более сильную модель, например:

- `DeepPavlov/rubert-base-cased`
- `ai-forever/ruBert-base`
- `xlm-roberta-base`

Начни с tiny-модели, убедись, что всё работает, затем меняй `MODEL_NAME`.

In [ ]:
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
import inspect

MODEL_NAME = "cointegrated/rubert-tiny2"
MAX_LENGTH = 384
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
USE_CLASS_WEIGHTS = False

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def make_hf_dataset(df: pd.DataFrame) -> Dataset:
    return Dataset.from_pandas(
        df[["text", "label_id"]].rename(columns={"label_id": "labels"}),
        preserve_index=False,
    )


def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )


hf_train = make_hf_dataset(train_df)
hf_valid = make_hf_dataset(valid_df)

tokenized_train = hf_train.map(tokenize_batch, batched=True, remove_columns=["text"])
tokenized_valid = hf_valid.map(tokenize_batch, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


def trainer_tokenizer_kwargs():
    trainer_params = inspect.signature(Trainer.__init__).parameters
    if "processing_class" in trainer_params:
        return {"processing_class": tokenizer}
    if "tokenizer" in trainer_params:
        return {"tokenizer": tokenizer}
    return {}


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    pred_ids = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, pred_ids),
        "macro_f1": f1_score(labels, pred_ids, average="macro"),
    }


class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits,
            labels,
            weight=self.class_weights.to(outputs.logits.device) if self.class_weights is not None else None,
        )
        return (loss, outputs) if return_outputs else loss


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_VALUES),
    id2label={i: str(ID_TO_LABEL[i]) for i in LABEL_IDS},
    label2id={str(label): label_id for label, label_id in LABEL_TO_ID.items()},
)

class_weights = None
if USE_CLASS_WEIGHTS:
    counts = np.bincount(train_df["label_id"], minlength=len(LABEL_VALUES))
    weights = counts.sum() / (len(LABEL_VALUES) * counts)
    class_weights = torch.tensor(weights, dtype=torch.float)
    print("Class weights:", class_weights)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "transformer_checkpoints"),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer_common_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    **trainer_tokenizer_kwargs(),
)

if USE_CLASS_WEIGHTS:
    trainer = WeightedTrainer(**trainer_common_kwargs, class_weights=class_weights)
else:
    trainer = Trainer(**trainer_common_kwargs)

trainer.train()
trainer.save_model(str(OUTPUT_DIR / "best_transformer_model"))
tokenizer.save_pretrained(str(OUTPUT_DIR / "best_transformer_model"))

In [ ]:
transformer_valid_output = trainer.predict(tokenized_valid)
transformer_valid_pred_ids = np.argmax(transformer_valid_output.predictions, axis=-1)
transformer_valid_pred_labels = decode_label_ids(transformer_valid_pred_ids)

transformer_accuracy = accuracy_score(valid_df["label_id"], transformer_valid_pred_ids)
print(f"Transformer validation accuracy: {transformer_accuracy:.4f}")
print(
    classification_report(
        valid_df["label_id"],
        transformer_valid_pred_ids,
        labels=LABEL_IDS,
        target_names=LABEL_NAMES,
        digits=4,
        zero_division=0,
    )
)

transformer_cm = pd.DataFrame(
    confusion_matrix(valid_df["label_id"], transformer_valid_pred_ids, labels=LABEL_IDS),
    index=[f"true_{x}" for x in LABEL_VALUES],
    columns=[f"pred_{x}" for x in LABEL_VALUES],
)
display(transformer_cm)

transformer_valid_out = valid_df.copy()
transformer_valid_out["predicted_relevance"] = transformer_valid_pred_labels
transformer_valid_out["is_correct"] = (
    transformer_valid_out["relevance"] == transformer_valid_out["predicted_relevance"]
)
transformer_valid_out.to_csv(
    OUTPUT_DIR / "transformer_valid_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)
transformer_valid_out.loc[~transformer_valid_out["is_correct"]].head(300).to_csv(
    OUTPUT_DIR / "transformer_errors_sample.csv",
    index=False,
    encoding="utf-8-sig",
)

## 3.1. Калибровка логитов на validation

Если модель почти не выбирает класс `0.1`, можно подобрать небольшие bias-поправки к логитам классов на validation. Это не использует eval и не меняет веса модели.

In [ ]:
def predict_with_bias(logits: np.ndarray, bias: np.ndarray) -> np.ndarray:
    return np.argmax(logits + bias.reshape(1, -1), axis=-1)


def tune_logit_bias(logits: np.ndarray, labels: np.ndarray, step: float = 0.1, limit: float = 2.0):
    # Adding the same constant to all logits changes nothing, so fix bias for class 0.0 at zero.
    grid = np.round(np.arange(-limit, limit + 1e-9, step), 4)
    best = {
        "accuracy": accuracy_score(labels, np.argmax(logits, axis=-1)),
        "bias": np.zeros(logits.shape[1], dtype=float),
    }
    for bias_01 in grid:
        for bias_10 in grid:
            bias = np.array([0.0, bias_01, bias_10], dtype=float)
            pred_ids = predict_with_bias(logits, bias)
            acc = accuracy_score(labels, pred_ids)
            if acc > best["accuracy"]:
                best = {"accuracy": acc, "bias": bias}
    return best


BEST_LOGIT_BIAS_INFO = tune_logit_bias(
    transformer_valid_output.predictions,
    valid_df["label_id"].to_numpy(),
    step=0.1,
    limit=2.0,
)
BEST_LOGIT_BIAS = BEST_LOGIT_BIAS_INFO["bias"]

print("Raw transformer accuracy:", f"{transformer_accuracy:.4f}")
print("Best calibrated accuracy:", f"{BEST_LOGIT_BIAS_INFO['accuracy']:.4f}")
print("Best logit bias:", BEST_LOGIT_BIAS.tolist())

calibrated_valid_pred_ids = predict_with_bias(transformer_valid_output.predictions, BEST_LOGIT_BIAS)
calibrated_valid_pred_labels = decode_label_ids(calibrated_valid_pred_ids)

print(
    classification_report(
        valid_df["label_id"],
        calibrated_valid_pred_ids,
        labels=LABEL_IDS,
        target_names=LABEL_NAMES,
        digits=4,
        zero_division=0,
    )
)

calibrated_cm = pd.DataFrame(
    confusion_matrix(valid_df["label_id"], calibrated_valid_pred_ids, labels=LABEL_IDS),
    index=[f"true_{x}" for x in LABEL_VALUES],
    columns=[f"pred_{x}" for x in LABEL_VALUES],
)
display(calibrated_cm)

calibrated_valid_out = valid_df.copy()
calibrated_valid_out["predicted_relevance"] = calibrated_valid_pred_labels
calibrated_valid_out["is_correct"] = (
    calibrated_valid_out["relevance"] == calibrated_valid_out["predicted_relevance"]
)
calibrated_valid_out.to_csv(
    OUTPUT_DIR / "transformer_calibrated_valid_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

(OUTPUT_DIR / "transformer_logit_bias.json").write_text(
    json.dumps(
        {
            "raw_accuracy": float(transformer_accuracy),
            "calibrated_accuracy": float(BEST_LOGIT_BIAS_INFO["accuracy"]),
            "bias": BEST_LOGIT_BIAS.tolist(),
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

## 3.2. Усиленный режим: ensemble трансформеров

Этот блок обучает несколько сильных моделей на одном train/valid split, сохраняет validation/eval logits, подбирает веса ансамбля и class-bias на validation, затем пишет `submission_ensemble.csv`.

Можно пропустить разделы `2`, `3` и `3.1`, если цель - только ensemble. Перед этим должны быть выполнены только подготовка, загрузка данных и split из раздела `1`.

По умолчанию включены две русские BERT-модели. `xlm-roberta-base` добавлен как опциональный третий участник: он медленнее, поэтому сначала лучше прогнать первые две.

In [ ]:
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
import inspect

RUN_STRONG_ENSEMBLE = True

STRONG_MODEL_CONFIGS = [
    {
        "name": "ai-forever/ruBert-base",
        "slug": "ai_forever_rubert_base",
        "enabled": True,
        "max_length": 384,
        "learning_rate": 2e-5,
        "epochs": 3,
        "train_batch_size": 8,
        "eval_batch_size": 16,
        "gradient_accumulation_steps": 2,
        "weight_decay": 0.01,
        "force_retrain": False,
    },
    {
        "name": "DeepPavlov/rubert-base-cased",
        "slug": "deeppavlov_rubert_base_cased",
        "enabled": True,
        "max_length": 384,
        "learning_rate": 2e-5,
        "epochs": 3,
        "train_batch_size": 8,
        "eval_batch_size": 16,
        "gradient_accumulation_steps": 2,
        "weight_decay": 0.01,
        "force_retrain": False,
    },
    {
        "name": "xlm-roberta-base",
        "slug": "xlm_roberta_base",
        "enabled": False,
        "max_length": 384,
        "learning_rate": 1.5e-5,
        "epochs": 3,
        "train_batch_size": 8,
        "eval_batch_size": 16,
        "gradient_accumulation_steps": 2,
        "weight_decay": 0.01,
        "force_retrain": False,
    },
]

ENSEMBLE_DIR = OUTPUT_DIR / "strong_ensemble"
ENSEMBLE_DIR.mkdir(parents=True, exist_ok=True)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    pred_ids = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, pred_ids),
        "macro_f1": f1_score(labels, pred_ids, average="macro"),
    }


def predict_with_bias(logits: np.ndarray, bias: np.ndarray) -> np.ndarray:
    return np.argmax(logits + bias.reshape(1, -1), axis=-1)


def tune_logit_bias(logits: np.ndarray, labels: np.ndarray, step: float = 0.1, limit: float = 2.0):
    # Adding the same constant to all logits changes nothing, so fix bias for class 0.0 at zero.
    grid = np.round(np.arange(-limit, limit + 1e-9, step), 4)
    best = {
        "accuracy": accuracy_score(labels, np.argmax(logits, axis=-1)),
        "bias": np.zeros(logits.shape[1], dtype=float),
    }
    for bias_01 in grid:
        for bias_10 in grid:
            bias = np.array([0.0, bias_01, bias_10], dtype=float)
            pred_ids = predict_with_bias(logits, bias)
            acc = accuracy_score(labels, pred_ids)
            if acc > best["accuracy"]:
                best = {"accuracy": acc, "bias": bias}
    return best


def training_args_compat(**kwargs):
    params = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" not in params and "evaluation_strategy" in params and "eval_strategy" in kwargs:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
    return TrainingArguments(**kwargs)


def trainer_processing_kwargs(tokenizer_obj):
    trainer_params = inspect.signature(Trainer.__init__).parameters
    if "processing_class" in trainer_params:
        return {"processing_class": tokenizer_obj}
    if "tokenizer" in trainer_params:
        return {"tokenizer": tokenizer_obj}
    return {}


def softmax_np(logits: np.ndarray) -> np.ndarray:
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)


def make_weight_grid(n_models: int, step: float = 0.1):
    values = np.round(np.arange(0.0, 1.0 + 1e-9, step), 4)

    def rec(prefix, remaining, slots_left):
        if slots_left == 1:
            yield prefix + [round(remaining, 4)]
            return
        for value in values:
            if value <= remaining + 1e-9:
                yield from rec(prefix + [float(value)], round(remaining - float(value), 4), slots_left - 1)

    yield from rec([], 1.0, n_models)


def combine_probabilities(logits_list, weights):
    probs = np.stack([softmax_np(logits) for logits in logits_list], axis=0)
    return np.tensordot(np.array(weights, dtype=float), probs, axes=(0, 0))


def tune_ensemble(valid_logits_list, labels, weight_step=0.1, bias_step=0.1, bias_limit=2.0):
    best = None
    for weights in make_weight_grid(len(valid_logits_list), step=weight_step):
        probs = combine_probabilities(valid_logits_list, weights)
        scores = np.log(np.clip(probs, 1e-12, 1.0))
        raw_pred = np.argmax(scores, axis=-1)
        raw_acc = accuracy_score(labels, raw_pred)
        bias_info = tune_logit_bias(scores, labels, step=bias_step, limit=bias_limit)
        item = {
            "weights": weights,
            "raw_accuracy": float(raw_acc),
            "accuracy": float(bias_info["accuracy"]),
            "bias": bias_info["bias"],
        }
        if best is None or item["accuracy"] > best["accuracy"]:
            best = item
    return best


def fit_or_load_ensemble_member(cfg):
    slug = cfg["slug"]
    model_dir = ENSEMBLE_DIR / slug
    valid_logits_path = ENSEMBLE_DIR / f"{slug}_valid_logits.npy"
    eval_logits_path = ENSEMBLE_DIR / f"{slug}_eval_logits.npy"
    score_path = ENSEMBLE_DIR / f"{slug}_score.json"

    if (
        valid_logits_path.exists()
        and eval_logits_path.exists()
        and score_path.exists()
        and not cfg.get("force_retrain", False)
    ):
        print(f"[load] {cfg['name']} logits")
        valid_logits = np.load(valid_logits_path)
        eval_logits = np.load(eval_logits_path)
        score = json.loads(score_path.read_text(encoding="utf-8"))
        score["valid_logits"] = valid_logits
        score["eval_logits"] = eval_logits
        return score

    print(f"[train] {cfg['name']}")
    tokenizer_local = AutoTokenizer.from_pretrained(cfg["name"])

    def tokenize_local(batch):
        return tokenizer_local(
            batch["text"],
            truncation=True,
            max_length=cfg.get("max_length", 384),
        )

    hf_train_local = Dataset.from_pandas(
        train_df[["text", "label_id"]].rename(columns={"label_id": "labels"}),
        preserve_index=False,
    )
    hf_valid_local = Dataset.from_pandas(
        valid_df[["text", "label_id"]].rename(columns={"label_id": "labels"}),
        preserve_index=False,
    )
    hf_eval_local = Dataset.from_pandas(eval_data[["text"]], preserve_index=False)

    tokenized_train_local = hf_train_local.map(tokenize_local, batched=True, remove_columns=["text"])
    tokenized_valid_local = hf_valid_local.map(tokenize_local, batched=True, remove_columns=["text"])
    tokenized_eval_local = hf_eval_local.map(tokenize_local, batched=True, remove_columns=["text"])

    collator_local = DataCollatorWithPadding(tokenizer=tokenizer_local)
    model_local = AutoModelForSequenceClassification.from_pretrained(
        cfg["name"],
        num_labels=len(LABEL_VALUES),
        id2label={i: str(ID_TO_LABEL[i]) for i in LABEL_IDS},
        label2id={str(label): label_id for label, label_id in LABEL_TO_ID.items()},
    )

    args_local = training_args_compat(
        output_dir=str(ENSEMBLE_DIR / f"{slug}_checkpoints"),
        learning_rate=cfg.get("learning_rate", 2e-5),
        per_device_train_batch_size=cfg.get("train_batch_size", 8),
        per_device_eval_batch_size=cfg.get("eval_batch_size", 16),
        gradient_accumulation_steps=cfg.get("gradient_accumulation_steps", 1),
        num_train_epochs=cfg.get("epochs", 3),
        weight_decay=cfg.get("weight_decay", 0.01),
        warmup_ratio=cfg.get("warmup_ratio", 0.06),
        eval_strategy="steps",
        eval_steps=cfg.get("eval_steps", 250),
        save_strategy="steps",
        save_steps=cfg.get("save_steps", 250),
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        logging_steps=50,
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=SEED,
        optim="adamw_torch",
    )

    trainer_local = Trainer(
        model=model_local,
        args=args_local,
        train_dataset=tokenized_train_local,
        eval_dataset=tokenized_valid_local,
        data_collator=collator_local,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        **trainer_processing_kwargs(tokenizer_local),
    )

    trainer_local.train()
    trainer_local.save_model(str(model_dir))
    tokenizer_local.save_pretrained(str(model_dir))

    valid_logits = trainer_local.predict(tokenized_valid_local).predictions
    eval_logits = trainer_local.predict(tokenized_eval_local).predictions

    np.save(valid_logits_path, valid_logits)
    np.save(eval_logits_path, eval_logits)

    raw_pred_ids = np.argmax(valid_logits, axis=-1)
    raw_acc = accuracy_score(valid_df["label_id"], raw_pred_ids)
    bias_info = tune_logit_bias(valid_logits, valid_df["label_id"].to_numpy(), step=0.1, limit=2.0)

    score = {
        "name": cfg["name"],
        "slug": slug,
        "raw_accuracy": float(raw_acc),
        "calibrated_accuracy": float(bias_info["accuracy"]),
        "bias": bias_info["bias"].tolist(),
        "valid_logits_path": str(valid_logits_path),
        "eval_logits_path": str(eval_logits_path),
        "model_dir": str(model_dir),
    }
    score_path.write_text(json.dumps(score, ensure_ascii=False, indent=2), encoding="utf-8")

    del trainer_local, model_local, tokenizer_local
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    score["valid_logits"] = valid_logits
    score["eval_logits"] = eval_logits
    return score


ENSEMBLE_RECORDS = []

if RUN_STRONG_ENSEMBLE:
    for cfg in STRONG_MODEL_CONFIGS:
        if not cfg.get("enabled", True):
            print(f"[skip] {cfg['name']}")
            continue
        ENSEMBLE_RECORDS.append(fit_or_load_ensemble_member(cfg))

    model_score_rows = [
        {
            "name": item["name"],
            "slug": item["slug"],
            "raw_accuracy": item["raw_accuracy"],
            "calibrated_accuracy": item["calibrated_accuracy"],
            "bias": item["bias"],
        }
        for item in ENSEMBLE_RECORDS
    ]
    model_scores_df = pd.DataFrame(model_score_rows)
    model_scores_df.to_csv(ENSEMBLE_DIR / "model_scores.csv", index=False, encoding="utf-8-sig")
    display(model_scores_df)
else:
    print("RUN_STRONG_ENSEMBLE=False. Set it to True to train the ensemble.")

In [ ]:
if RUN_STRONG_ENSEMBLE and ENSEMBLE_RECORDS:
    valid_logits_list = [item["valid_logits"] for item in ENSEMBLE_RECORDS]
    eval_logits_list = [item["eval_logits"] for item in ENSEMBLE_RECORDS]
    y_valid = valid_df["label_id"].to_numpy()

    BEST_ENSEMBLE_INFO = tune_ensemble(
        valid_logits_list,
        y_valid,
        weight_step=0.1,
        bias_step=0.1,
        bias_limit=2.0,
    )
    BEST_ENSEMBLE_WEIGHTS = np.array(BEST_ENSEMBLE_INFO["weights"], dtype=float)
    BEST_ENSEMBLE_BIAS = np.array(BEST_ENSEMBLE_INFO["bias"], dtype=float)

    ensemble_valid_probs = combine_probabilities(valid_logits_list, BEST_ENSEMBLE_WEIGHTS)
    ensemble_valid_scores = np.log(np.clip(ensemble_valid_probs, 1e-12, 1.0))
    ensemble_valid_pred_ids = predict_with_bias(ensemble_valid_scores, BEST_ENSEMBLE_BIAS)
    ensemble_valid_pred_labels = decode_label_ids(ensemble_valid_pred_ids)

    print("Best ensemble raw accuracy:", f"{BEST_ENSEMBLE_INFO['raw_accuracy']:.4f}")
    print("Best ensemble calibrated accuracy:", f"{BEST_ENSEMBLE_INFO['accuracy']:.4f}")
    print("Models:", [item["name"] for item in ENSEMBLE_RECORDS])
    print("Weights:", BEST_ENSEMBLE_WEIGHTS.tolist())
    print("Bias:", BEST_ENSEMBLE_BIAS.tolist())

    print(
        classification_report(
            valid_df["label_id"],
            ensemble_valid_pred_ids,
            labels=LABEL_IDS,
            target_names=LABEL_NAMES,
            digits=4,
            zero_division=0,
        )
    )

    ensemble_cm = pd.DataFrame(
        confusion_matrix(valid_df["label_id"], ensemble_valid_pred_ids, labels=LABEL_IDS),
        index=[f"true_{x}" for x in LABEL_VALUES],
        columns=[f"pred_{x}" for x in LABEL_VALUES],
    )
    display(ensemble_cm)

    ensemble_valid_out = valid_df.copy()
    ensemble_valid_out["predicted_relevance"] = ensemble_valid_pred_labels
    ensemble_valid_out["is_correct"] = (
        ensemble_valid_out["relevance"] == ensemble_valid_out["predicted_relevance"]
    )
    ensemble_valid_out.to_csv(
        OUTPUT_DIR / "ensemble_valid_predictions.csv",
        index=False,
        encoding="utf-8-sig",
    )
    ensemble_valid_out.loc[~ensemble_valid_out["is_correct"]].head(300).to_csv(
        OUTPUT_DIR / "ensemble_errors_sample.csv",
        index=False,
        encoding="utf-8-sig",
    )

    ensemble_eval_probs = combine_probabilities(eval_logits_list, BEST_ENSEMBLE_WEIGHTS)
    ensemble_eval_scores = np.log(np.clip(ensemble_eval_probs, 1e-12, 1.0))
    ensemble_eval_pred_ids = predict_with_bias(ensemble_eval_scores, BEST_ENSEMBLE_BIAS)
    ensemble_eval_pred_labels = decode_label_ids(ensemble_eval_pred_ids)

    ensemble_eval_out = eval_data.copy()
    ensemble_eval_out["predicted_relevance"] = ensemble_eval_pred_labels
    ensemble_eval_out.to_csv(
        OUTPUT_DIR / "ensemble_eval_predictions.csv",
        index=False,
        encoding="utf-8-sig",
    )

    submission_ensemble = pd.DataFrame(
        {
            "permalink": eval_data.get("permalink"),
            "Text": eval_data.get("Text"),
            "relevance": ensemble_eval_pred_labels,
        }
    )
    submission_ensemble.to_csv(
        OUTPUT_DIR / "submission_ensemble.csv",
        index=False,
        encoding="utf-8-sig",
    )

    ensemble_config = {
        "models": [
            {
                "name": item["name"],
                "slug": item["slug"],
                "raw_accuracy": item["raw_accuracy"],
                "calibrated_accuracy": item["calibrated_accuracy"],
                "single_model_bias": item["bias"],
            }
            for item in ENSEMBLE_RECORDS
        ],
        "ensemble_raw_accuracy": float(BEST_ENSEMBLE_INFO["raw_accuracy"]),
        "ensemble_calibrated_accuracy": float(BEST_ENSEMBLE_INFO["accuracy"]),
        "weights": BEST_ENSEMBLE_WEIGHTS.tolist(),
        "bias": BEST_ENSEMBLE_BIAS.tolist(),
    }
    (ENSEMBLE_DIR / "ensemble_config.json").write_text(
        json.dumps(ensemble_config, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    display(ensemble_eval_out["predicted_relevance"].value_counts().sort_index())
    print("Saved:", OUTPUT_DIR / "submission_ensemble.csv")
    print("Saved:", ENSEMBLE_DIR / "ensemble_config.json")
else:
    print("No ensemble predictions. Run the previous cell with RUN_STRONG_ENSEMBLE=True.")

## 3.3. Validation subset для LLM-агента

Агента запускаем не на всех примерах, а на спорных случаях: низкая уверенность, маленький margin между top-1/top-2, предсказание `0.1` или расхождение raw/calibrated предсказаний.

Файл `agent_validation_cases.jsonl` содержит true labels, потому что это validation subset из train. На eval этот механизм не используем для калибровки.

In [ ]:
def softmax_np(logits: np.ndarray) -> np.ndarray:
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)


def json_safe(value):
    if value is None:
        return None
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    try:
        if pd.isna(value):
            return None
    except TypeError:
        pass
    return value


AGENT_CASE_LIMIT = 300
AGENT_CONFIDENCE_THRESHOLD = 0.72
AGENT_MARGIN_THRESHOLD = 0.18

if "ensemble_valid_scores" in globals() and "BEST_ENSEMBLE_BIAS" in globals():
    agent_selector_base = "strong_ensemble"
    raw_logits = ensemble_valid_scores
    calibration_bias = BEST_ENSEMBLE_BIAS
else:
    agent_selector_base = "single_transformer"
    raw_logits = transformer_valid_output.predictions
    calibration_bias = BEST_LOGIT_BIAS

raw_pred_ids = np.argmax(raw_logits, axis=-1)
calibrated_logits = raw_logits + calibration_bias.reshape(1, -1)
calibrated_probs = softmax_np(calibrated_logits)
calibrated_pred_ids = np.argmax(calibrated_logits, axis=-1)

sorted_probs = np.sort(calibrated_probs, axis=1)
confidence = sorted_probs[:, -1]
margin = sorted_probs[:, -1] - sorted_probs[:, -2]

agent_pool = valid_df.copy()
agent_pool["case_id"] = [f"valid_{i}_{row.permalink}" for i, row in agent_pool.iterrows()]
agent_pool["raw_transformer_predicted_relevance"] = decode_label_ids(raw_pred_ids)
agent_pool["transformer_predicted_relevance"] = decode_label_ids(calibrated_pred_ids)
agent_pool["transformer_confidence"] = confidence
agent_pool["transformer_margin"] = margin
agent_pool["raw_calibrated_disagree"] = raw_pred_ids != calibrated_pred_ids
agent_pool["agent_selector_base"] = agent_selector_base
agent_pool["is_uncertain"] = (
    (agent_pool["transformer_predicted_relevance"] == 0.1)
    | (agent_pool["transformer_confidence"] < AGENT_CONFIDENCE_THRESHOLD)
    | (agent_pool["transformer_margin"] < AGENT_MARGIN_THRESHOLD)
    | agent_pool["raw_calibrated_disagree"]
)

agent_cases_df = (
    agent_pool.loc[agent_pool["is_uncertain"]]
    .sort_values(["transformer_margin", "transformer_confidence"], ascending=[True, True])
    .head(AGENT_CASE_LIMIT)
    .reset_index(drop=True)
)

agent_fields = [
    "case_id",
    "Text",
    "name",
    "normalized_main_rubric_name_ru",
    "address",
    "prices_summarized",
    "reviews_summarized",
    "relevance",
    "raw_transformer_predicted_relevance",
    "transformer_predicted_relevance",
    "transformer_confidence",
    "transformer_margin",
    "raw_calibrated_disagree",
    "agent_selector_base",
]

agent_cases_path = OUTPUT_DIR / "agent_validation_cases.jsonl"
with agent_cases_path.open("w", encoding="utf-8") as f:
    for _, row in agent_cases_df.iterrows():
        item = {field: json_safe(row.get(field)) for field in agent_fields}
        item["true_relevance"] = item.pop("relevance")
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Agent validation cases:", len(agent_cases_df))
print("Agent selector base:", agent_selector_base)
print("Saved:", agent_cases_path)
print("True distribution:")
display(agent_cases_df["relevance"].value_counts().sort_index())
print("Transformer prediction distribution on agent subset:")
display(agent_cases_df["transformer_predicted_relevance"].value_counts().sort_index())
display(
    agent_cases_df[
        [
            "Text",
            "name",
            "normalized_main_rubric_name_ru",
            "relevance",
            "transformer_predicted_relevance",
            "transformer_confidence",
            "transformer_margin",
        ]
    ].head(10)
)

fewshot_fields = [
    "Text",
    "name",
    "normalized_main_rubric_name_ru",
    "address",
    "prices_summarized",
    "reviews_summarized",
    "relevance",
    "permalink",
]

fewshot_path = OUTPUT_DIR / "agent_fewshot_train_examples.jsonl"
with fewshot_path.open("w", encoding="utf-8") as f:
    for _, row in train_data[fewshot_fields].iterrows():
        item = {field: json_safe(row.get(field)) for field in fewshot_fields}
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

agent_eval_fields = [
    "case_id",
    "Text",
    "name",
    "normalized_main_rubric_name_ru",
    "address",
    "prices_summarized",
    "reviews_summarized",
    "permalink",
]

agent_eval_df = eval_data.copy().reset_index(drop=True)
agent_eval_df["case_id"] = [f"eval_{i}_{row.permalink}" for i, row in agent_eval_df.iterrows()]
agent_eval_path = OUTPUT_DIR / "agent_eval_cases.jsonl"
with agent_eval_path.open("w", encoding="utf-8") as f:
    for _, row in agent_eval_df.iterrows():
        item = {field: json_safe(row.get(field)) for field in agent_eval_fields}
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Few-shot train examples:", len(train_data), "Saved:", fewshot_path)
print("Agent eval cases without labels:", len(agent_eval_df), "Saved:", agent_eval_path)

## 3.4. LLM/search agent runner

Runner ниже работает с OpenAI-compatible API. Можно использовать OpenAI напрямую или совместимый gateway.

Минимальные env-переменные:

```python
os.environ["ROUTERAI_API_KEY"] = "..."
os.environ["ROUTERAI_BASE_URL"] = "https://routerai.ru/api/v1"
os.environ["AGENT_MODEL"] = "deepseek/deepseek-v4-pro"
```

Для OpenAI-compatible gateway можно добавить:

```python
Path("/content/.env").write_text(
    "ROUTERAI_API_KEY=...\n"
    "ROUTERAI_BASE_URL=https://routerai.ru/api/v1\n"
    "AGENT_MODEL=deepseek/deepseek-v4-pro\n"
    "SEARCH_PROVIDER=none\n",
    encoding="utf-8",
)
```

Поиск опционален:

- без поиска: `SEARCH_PROVIDER=none`;
- Serper: `SEARCH_PROVIDER=serper`, `SERPER_API_KEY=...`;
- Tavily: `SEARCH_PROVIDER=tavily`, `TAVILY_API_KEY=...`.

In [ ]:
agent_runner_code = "import argparse\nimport csv\nimport json\nimport os\nimport re\nimport time\nimport urllib.error\nimport urllib.request\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, List, Optional\n\n\nLABEL_VALUES = [0.0, 0.1, 1.0]\nDEFAULT_AGENT_MODEL = \"deepseek/deepseek-v4-pro\"\nDEFAULT_BASE_URL = \"https://routerai.ru/api/v1\"\n\n\nSYSTEM_PROMPT = \"\"\"\nТы оцениваешь релевантность организаций на Яндекс.Картах широким пользовательским запросам.\n\nКлассы:\n- 1.0: организация явно подходит запросу; услуга, категория или важное свойство подтверждены.\n- 0.1: организация частично подходит: близкая категория, но нет нужного свойства; услуга смежная; релевантность слабая или неоднозначная.\n- 0.0: организация не подходит запросу или нет достаточных оснований считать её подходящей.\n\nБудь строгим. Если пользователь ищет конкретное свойство (\"веранда\", \"живая музыка\", \"мероприятия\", \"бесплатное занятие\"), не ставь 1.0 без прямого или очень сильного подтверждения.\nВозвращай только JSON без markdown.\n\"\"\".strip()\n\n\nPLAN_PROMPT = \"\"\"\nОцени карточку организации по локальным данным и реши, нужен ли внешний поиск.\n\nВерни JSON строго такого вида:\n{\n  \"local_label\": 0.0,\n  \"local_confidence\": 0.0,\n  \"needs_search\": true,\n  \"search_queries\": [\"...\"],\n  \"evidence\": \"краткие факты из локальных данных\",\n  \"explanation\": \"краткое объяснение\"\n}\n\nПравила:\n- local_label должен быть одним из 0.0, 0.1, 1.0.\n- local_confidence от 0 до 1.\n- needs_search=true, если локальных данных не хватает для проверки важного свойства запроса.\n- search_queries максимум 2, на русском, с названием/адресом организации и важным свойством из запроса.\n\"\"\".strip()\n\n\nFINAL_PROMPT = \"\"\"\nПрими финальное решение о релевантности организации запросу.\n\nВерни JSON строго такого вида:\n{\n  \"label\": 0.0,\n  \"confidence\": 0.0,\n  \"used_search\": false,\n  \"evidence\": \"краткие факты\",\n  \"explanation\": \"почему выбран этот класс\"\n}\n\nlabel должен быть одним из 0.0, 0.1, 1.0.\nconfidence от 0 до 1.\n\"\"\".strip()\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=\"Run a cached LLM/search agent on relevance cases.\")\n    parser.add_argument(\"--cases\", required=True, help=\"JSONL with validation/eval cases.\")\n    parser.add_argument(\"--out\", required=True, help=\"JSONL output cache/results.\")\n    parser.add_argument(\"--env-file\", default=\".env\", help=\"Optional dotenv-style file with API settings.\")\n    parser.add_argument(\"--model\", default=None)\n    parser.add_argument(\"--base-url\", default=None)\n    parser.add_argument(\"--api-key\", default=None)\n    parser.add_argument(\n        \"--search-provider\",\n        choices=[\"none\", \"serper\", \"tavily\"],\n        default=None,\n    )\n    parser.add_argument(\"--serper-api-key\", default=None)\n    parser.add_argument(\"--tavily-api-key\", default=None)\n    parser.add_argument(\"--limit\", type=int, default=0, help=\"0 means all cases.\")\n    parser.add_argument(\"--max-search-queries\", type=int, default=2)\n    parser.add_argument(\"--max-search-results\", type=int, default=4)\n    parser.add_argument(\"--fewshot-path\", default=\"\", help=\"Optional JSONL with labeled examples for few-shot context.\")\n    parser.add_argument(\"--fewshot-k\", type=int, default=0, help=\"Number of similar labeled examples to add.\")\n    parser.add_argument(\"--temperature\", type=float, default=0.0)\n    parser.add_argument(\"--sleep\", type=float, default=0.0)\n    parser.add_argument(\"--submission-csv\", default=\"\", help=\"Optional CSV path with permalink, Text, relevance.\")\n    parser.add_argument(\"--overwrite\", action=\"store_true\")\n    parser.add_argument(\"--evaluate\", action=\"store_true\")\n    return parser.parse_args()\n\n\ndef load_env_file(path: Path) -> None:\n    \"\"\"Load simple KEY=VALUE pairs without adding a dependency on python-dotenv.\"\"\"\n    if not path.exists():\n        return\n\n    with path.open(\"r\", encoding=\"utf-8-sig\") as f:\n        for line in f:\n            stripped = line.strip()\n            if not stripped or stripped.startswith(\"#\") or \"=\" not in stripped:\n                continue\n            key, value = stripped.split(\"=\", 1)\n            key = key.strip()\n            if key.startswith(\"export \"):\n                key = key[len(\"export \") :].strip()\n            if not re.fullmatch(r\"[A-Za-z_][A-Za-z0-9_]*\", key):\n                continue\n            value = value.strip()\n            if len(value) >= 2 and value[0] == value[-1] and value[0] in (\"'\", '\"'):\n                value = value[1:-1]\n            os.environ.setdefault(key, value)\n\n\ndef configure_args(args: argparse.Namespace) -> argparse.Namespace:\n    load_env_file(Path(args.env_file).expanduser())\n    args.model = args.model or os.getenv(\"AGENT_MODEL\") or DEFAULT_AGENT_MODEL\n    args.base_url = (\n        args.base_url\n        or os.getenv(\"OPENAI_BASE_URL\")\n        or os.getenv(\"ROUTERAI_BASE_URL\")\n        or DEFAULT_BASE_URL\n    )\n    args.api_key = args.api_key or os.getenv(\"OPENAI_API_KEY\") or os.getenv(\"ROUTERAI_API_KEY\")\n    args.search_provider = str(args.search_provider or os.getenv(\"SEARCH_PROVIDER\") or \"none\").lower()\n    if args.search_provider not in {\"none\", \"serper\", \"tavily\"}:\n        raise ValueError(\"SEARCH_PROVIDER must be one of: none, serper, tavily.\")\n    args.serper_api_key = args.serper_api_key or os.getenv(\"SERPER_API_KEY\")\n    args.tavily_api_key = args.tavily_api_key or os.getenv(\"TAVILY_API_KEY\")\n    return args\n\n\ndef read_jsonl(path: Path) -> List[Dict[str, Any]]:\n    rows = []\n    with path.open(\"r\", encoding=\"utf-8-sig\") as f:\n        for line_no, line in enumerate(f, 1):\n            line = line.strip()\n            if not line:\n                continue\n            try:\n                rows.append(json.loads(line))\n            except json.JSONDecodeError as exc:\n                raise ValueError(f\"Cannot parse {path} line {line_no}: {exc}\") from exc\n    return rows\n\n\ndef append_jsonl(path: Path, row: Dict[str, Any]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\"a\", encoding=\"utf-8\") as f:\n        f.write(json.dumps(row, ensure_ascii=False) + \"\\n\")\n\n\ndef existing_case_ids(path: Path) -> set:\n    if not path.exists():\n        return set()\n    ids = set()\n    for row in read_jsonl(path):\n        case_id = row.get(\"case_id\")\n        if case_id is not None:\n            ids.add(str(case_id))\n    return ids\n\n\ndef ensure_case_ids(cases: List[Dict[str, Any]]) -> List[Dict[str, Any]]:\n    normalized = []\n    for idx, case in enumerate(cases):\n        item = dict(case)\n        if not item.get(\"case_id\"):\n            permalink = item.get(\"permalink\")\n            if permalink is not None:\n                item[\"case_id\"] = f\"case_{idx}_{permalink}\"\n            else:\n                item[\"case_id\"] = f\"case_{idx}\"\n        normalized.append(item)\n    return normalized\n\n\ndef http_json(url: str, headers: Dict[str, str], payload: Dict[str, Any], timeout: int = 90) -> Dict[str, Any]:\n    data = json.dumps(payload, ensure_ascii=False).encode(\"utf-8\")\n    request = urllib.request.Request(url, data=data, headers=headers, method=\"POST\")\n    try:\n        with urllib.request.urlopen(request, timeout=timeout) as response:\n            return json.loads(response.read().decode(\"utf-8\"))\n    except urllib.error.HTTPError as exc:\n        body = exc.read().decode(\"utf-8\", errors=\"replace\")\n        raise RuntimeError(f\"HTTP {exc.code} from {url}: {body[:1000]}\") from exc\n\n\ndef call_llm(\n    messages: List[Dict[str, str]],\n    model: str,\n    base_url: str,\n    api_key: Optional[str],\n    temperature: float,\n) -> Dict[str, Any]:\n    if not api_key:\n        raise RuntimeError(\"OPENAI_API_KEY or ROUTERAI_API_KEY is required for LLM calls.\")\n\n    url = base_url.rstrip(\"/\") + \"/chat/completions\"\n    headers = {\n        \"Authorization\": f\"Bearer {api_key}\",\n        \"Content-Type\": \"application/json\",\n    }\n    payload = {\n        \"model\": model,\n        \"messages\": messages,\n        \"temperature\": temperature,\n        \"response_format\": {\"type\": \"json_object\"},\n    }\n    response = http_json(url, headers, payload)\n    content = response[\"choices\"][0][\"message\"][\"content\"]\n    try:\n        parsed = json.loads(content)\n    except json.JSONDecodeError:\n        # Some OpenAI-compatible gateways ignore response_format.\n        start = content.find(\"{\")\n        end = content.rfind(\"}\")\n        if start == -1 or end == -1 or end <= start:\n            raise\n        parsed = json.loads(content[start : end + 1])\n    return {\"parsed\": parsed, \"raw_response\": response}\n\n\ndef case_to_text(case: Dict[str, Any]) -> str:\n    fields = [\n        (\"Запрос\", case.get(\"Text\")),\n        (\"Название\", case.get(\"name\")),\n        (\"Рубрика\", case.get(\"normalized_main_rubric_name_ru\") or case.get(\"rubric\")),\n        (\"Адрес\", case.get(\"address\")),\n        (\"Товары и услуги\", case.get(\"prices_summarized\")),\n        (\"Отзывы и описание\", case.get(\"reviews_summarized\")),\n        (\"Предсказание трансформера\", case.get(\"transformer_predicted_relevance\")),\n        (\"Уверенность трансформера\", case.get(\"transformer_confidence\")),\n        (\"Отрыв top-1 от top-2\", case.get(\"transformer_margin\")),\n    ]\n    return \"\\n\".join(f\"{name}: {'' if value is None else value}\" for name, value in fields)\n\n\ndef label_from_case(case: Dict[str, Any]) -> Optional[float]:\n    for key in (\"true_relevance\", \"relevance\", \"relevance_new\", \"label\"):\n        label = normalize_label(case.get(key))\n        if label is not None:\n            return label\n    return None\n\n\ndef token_set(case: Dict[str, Any]) -> set:\n    text = \" \".join(\n        str(case.get(key) or \"\")\n        for key in (\n            \"Text\",\n            \"name\",\n            \"normalized_main_rubric_name_ru\",\n            \"rubric\",\n            \"address\",\n            \"prices_summarized\",\n            \"reviews_summarized\",\n        )\n    ).lower()\n    return set(re.findall(r\"[\\wа-яё]+\", text, flags=re.IGNORECASE))\n\n\ndef same_case(left: Dict[str, Any], right: Dict[str, Any]) -> bool:\n    if left.get(\"case_id\") and right.get(\"case_id\") and str(left[\"case_id\"]) == str(right[\"case_id\"]):\n        return True\n    return (\n        left.get(\"permalink\") is not None\n        and right.get(\"permalink\") is not None\n        and str(left.get(\"permalink\")) == str(right.get(\"permalink\"))\n        and str(left.get(\"Text\")) == str(right.get(\"Text\"))\n    )\n\n\ndef select_fewshot_examples(case: Dict[str, Any], examples: List[Dict[str, Any]], k: int) -> List[Dict[str, Any]]:\n    if k <= 0 or not examples:\n        return []\n    case_tokens = token_set(case)\n    scored = []\n    for example in examples:\n        label = label_from_case(example)\n        if label is None or same_case(case, example):\n            continue\n        example_tokens = token_set(example)\n        overlap = len(case_tokens & example_tokens)\n        if overlap == 0:\n            continue\n        union = len(case_tokens | example_tokens) or 1\n        score = overlap / union\n        scored.append((score, overlap, example))\n    scored.sort(key=lambda item: (item[0], item[1]), reverse=True)\n    return [item[2] for item in scored[:k]]\n\n\ndef fewshot_to_text(examples: List[Dict[str, Any]]) -> str:\n    if not examples:\n        return \"Few-shot примеры не используются.\"\n    chunks = []\n    for idx, example in enumerate(examples, 1):\n        label = label_from_case(example)\n        chunks.append(\n            \"\\n\".join(\n                [\n                    f\"Пример {idx}:\",\n                    f\"Запрос: {example.get('Text', '')}\",\n                    f\"Название: {example.get('name', '')}\",\n                    f\"Рубрика: {example.get('normalized_main_rubric_name_ru') or example.get('rubric') or ''}\",\n                    f\"Адрес: {example.get('address', '')}\",\n                    f\"Правильная метка: {label}\",\n                ]\n            )\n        )\n    return \"\\n\\n\".join(chunks)\n\n\ndef normalize_label(value: Any) -> Optional[float]:\n    try:\n        label = float(value)\n    except (TypeError, ValueError):\n        return None\n    return label if label in LABEL_VALUES else None\n\n\ndef search_serper(query: str, api_key: str, max_results: int) -> List[Dict[str, str]]:\n    response = http_json(\n        \"https://google.serper.dev/search\",\n        headers={\"X-API-KEY\": api_key, \"Content-Type\": \"application/json\"},\n        payload={\"q\": query, \"hl\": \"ru\", \"gl\": \"ru\", \"num\": max_results},\n        timeout=45,\n    )\n    results = []\n    for item in response.get(\"organic\", [])[:max_results]:\n        results.append(\n            {\n                \"title\": item.get(\"title\", \"\"),\n                \"url\": item.get(\"link\", \"\"),\n                \"snippet\": item.get(\"snippet\", \"\"),\n            }\n        )\n    return results\n\n\ndef search_tavily(query: str, api_key: str, max_results: int) -> List[Dict[str, str]]:\n    response = http_json(\n        \"https://api.tavily.com/search\",\n        headers={\"Content-Type\": \"application/json\"},\n        payload={\n            \"api_key\": api_key,\n            \"query\": query,\n            \"search_depth\": \"basic\",\n            \"max_results\": max_results,\n            \"include_answer\": False,\n        },\n        timeout=45,\n    )\n    results = []\n    for item in response.get(\"results\", [])[:max_results]:\n        results.append(\n            {\n                \"title\": item.get(\"title\", \"\"),\n                \"url\": item.get(\"url\", \"\"),\n                \"snippet\": item.get(\"content\", \"\"),\n            }\n        )\n    return results\n\n\ndef run_search(\n    queries: Iterable[str],\n    provider: str,\n    serper_api_key: Optional[str],\n    tavily_api_key: Optional[str],\n    max_results: int,\n) -> List[Dict[str, Any]]:\n    search_results = []\n    if provider == \"none\":\n        return search_results\n\n    for query in queries:\n        query = str(query).strip()\n        if not query:\n            continue\n        if provider == \"serper\":\n            if not serper_api_key:\n                raise RuntimeError(\"SERPER_API_KEY is required for search_provider=serper.\")\n            results = search_serper(query, serper_api_key, max_results)\n        elif provider == \"tavily\":\n            if not tavily_api_key:\n                raise RuntimeError(\"TAVILY_API_KEY is required for search_provider=tavily.\")\n            results = search_tavily(query, tavily_api_key, max_results)\n        else:\n            raise ValueError(f\"Unknown search provider: {provider}\")\n        search_results.append({\"query\": query, \"results\": results})\n    return search_results\n\n\ndef search_results_to_text(search_results: List[Dict[str, Any]]) -> str:\n    if not search_results:\n        return \"Внешний поиск не использовался или не дал результатов.\"\n    chunks = []\n    for block in search_results:\n        chunks.append(f\"Запрос поиска: {block['query']}\")\n        for idx, item in enumerate(block.get(\"results\", []), 1):\n            chunks.append(\n                f\"{idx}. {item.get('title', '')}\\n\"\n                f\"URL: {item.get('url', '')}\\n\"\n                f\"Snippet: {item.get('snippet', '')}\"\n            )\n    return \"\\n\\n\".join(chunks)\n\n\ndef run_case(case: Dict[str, Any], args: argparse.Namespace) -> Dict[str, Any]:\n    case_text = case_to_text(case)\n    fewshot_examples = select_fewshot_examples(\n        case,\n        getattr(args, \"fewshot_examples\", []),\n        getattr(args, \"fewshot_k\", 0),\n    )\n    fewshot_text = fewshot_to_text(fewshot_examples)\n    plan_messages = [\n        {\"role\": \"system\", \"content\": SYSTEM_PROMPT},\n        {\n            \"role\": \"user\",\n            \"content\": PLAN_PROMPT + \"\\n\\nПохожие размеченные примеры:\\n\" + fewshot_text + \"\\n\\nКарточка:\\n\" + case_text,\n        },\n    ]\n    plan = call_llm(\n        plan_messages,\n        model=args.model,\n        base_url=args.base_url,\n        api_key=args.api_key,\n        temperature=args.temperature,\n    )\n    plan_json = plan[\"parsed\"]\n    queries = plan_json.get(\"search_queries\", []) or []\n    if not isinstance(queries, list):\n        queries = []\n    queries = queries[: args.max_search_queries]\n\n    search_results = []\n    if plan_json.get(\"needs_search\") and args.search_provider != \"none\":\n        search_results = run_search(\n            queries,\n            provider=args.search_provider,\n            serper_api_key=args.serper_api_key,\n            tavily_api_key=args.tavily_api_key,\n            max_results=args.max_search_results,\n        )\n\n    final_messages = [\n        {\"role\": \"system\", \"content\": SYSTEM_PROMPT},\n        {\n            \"role\": \"user\",\n            \"content\": (\n                FINAL_PROMPT\n                + \"\\n\\nКарточка:\\n\"\n                + case_text\n                + \"\\n\\nПохожие размеченные примеры:\\n\"\n                + fewshot_text\n                + \"\\n\\nПредварительная оценка:\\n\"\n                + json.dumps(plan_json, ensure_ascii=False, indent=2)\n                + \"\\n\\nРезультаты поиска:\\n\"\n                + search_results_to_text(search_results)\n            ),\n        },\n    ]\n    final = call_llm(\n        final_messages,\n        model=args.model,\n        base_url=args.base_url,\n        api_key=args.api_key,\n        temperature=args.temperature,\n    )\n    final_json = final[\"parsed\"]\n    label = normalize_label(final_json.get(\"label\"))\n\n    return {\n        \"case_id\": str(case.get(\"case_id\")),\n        \"permalink\": case.get(\"permalink\"),\n        \"Text\": case.get(\"Text\"),\n        \"label\": label,\n        \"true_relevance\": case.get(\"true_relevance\"),\n        \"transformer_predicted_relevance\": case.get(\"transformer_predicted_relevance\"),\n        \"transformer_confidence\": case.get(\"transformer_confidence\"),\n        \"transformer_margin\": case.get(\"transformer_margin\"),\n        \"used_search\": bool(search_results),\n        \"search_provider\": args.search_provider,\n        \"search_results\": search_results,\n        \"fewshot_examples\": [\n            {\n                \"Text\": item.get(\"Text\"),\n                \"name\": item.get(\"name\"),\n                \"rubric\": item.get(\"normalized_main_rubric_name_ru\") or item.get(\"rubric\"),\n                \"label\": label_from_case(item),\n            }\n            for item in fewshot_examples\n        ],\n        \"plan\": plan_json,\n        \"final\": final_json,\n    }\n\n\ndef evaluate(cases: List[Dict[str, Any]], results_path: Path) -> None:\n    results = read_jsonl(results_path)\n    case_by_id = {str(case.get(\"case_id\")): case for case in cases}\n    joined = []\n    for result in results:\n        case = case_by_id.get(str(result.get(\"case_id\")))\n        if not case:\n            continue\n        y_true = normalize_label(case.get(\"true_relevance\"))\n        y_agent = normalize_label(result.get(\"label\"))\n        y_base = normalize_label(case.get(\"transformer_predicted_relevance\"))\n        if y_true is None or y_agent is None:\n            continue\n        joined.append((y_true, y_agent, y_base))\n\n    if not joined:\n        print(\"No completed labeled results to evaluate.\")\n        return\n\n    total = len(joined)\n    agent_acc = sum(y_true == y_agent for y_true, y_agent, _ in joined) / total\n    base_acc = sum(y_true == y_base for y_true, _, y_base in joined if y_base is not None) / total\n    print(f\"Completed labeled cases: {total}\")\n    print(f\"Agent accuracy on completed cases: {agent_acc:.4f}\")\n    print(f\"Transformer accuracy on same cases: {base_acc:.4f}\")\n\n    print(\"\\nPer-class counts (true, agent):\")\n    counts: Dict[str, int] = {}\n    for y_true, y_agent, _ in joined:\n        key = f\"{y_true}->{y_agent}\"\n        counts[key] = counts.get(key, 0) + 1\n    for key, value in sorted(counts.items()):\n        print(f\"{key}: {value}\")\n\n\ndef write_submission_csv(results_path: Path, submission_path: Path) -> None:\n    results = read_jsonl(results_path)\n    submission_path.parent.mkdir(parents=True, exist_ok=True)\n    with submission_path.open(\"w\", encoding=\"utf-8-sig\", newline=\"\") as f:\n        writer = csv.DictWriter(f, fieldnames=[\"permalink\", \"Text\", \"relevance\"])\n        writer.writeheader()\n        for row in results:\n            label = normalize_label(row.get(\"label\"))\n            if label is None:\n                continue\n            writer.writerow(\n                {\n                    \"permalink\": row.get(\"permalink\"),\n                    \"Text\": row.get(\"Text\"),\n                    \"relevance\": label,\n                }\n            )\n    print(f\"Saved submission CSV: {submission_path}\")\n\n\ndef main() -> None:\n    args = configure_args(parse_args())\n    cases_path = Path(args.cases)\n    out_path = Path(args.out)\n    cases = ensure_case_ids(read_jsonl(cases_path))\n    if args.limit:\n        cases = cases[: args.limit]\n    args.fewshot_examples = read_jsonl(Path(args.fewshot_path)) if args.fewshot_path else []\n\n    done_ids = set() if args.overwrite else existing_case_ids(out_path)\n    print(f\"Loaded cases: {len(cases)}\")\n    print(f\"Already completed: {len(done_ids)}\")\n\n    for idx, case in enumerate(cases, 1):\n        case_id = str(case.get(\"case_id\"))\n        if case_id in done_ids:\n            continue\n        print(f\"[{idx}/{len(cases)}] case_id={case_id}\")\n        try:\n            result = run_case(case, args)\n        except Exception as exc:\n            result = {\n                \"case_id\": case_id,\n                \"label\": None,\n                \"error\": repr(exc),\n            }\n        append_jsonl(out_path, result)\n        if args.sleep:\n            time.sleep(args.sleep)\n\n    if args.evaluate:\n        evaluate(cases, out_path)\n\n    if args.submission_csv:\n        write_submission_csv(out_path, Path(args.submission_csv))\n\n\nif __name__ == \"__main__\":\n    main()\n"
Path("/content/agent_runner.py").write_text(agent_runner_code, encoding="utf-8")
print("Wrote /content/agent_runner.py")

In [ ]:
import subprocess

# Запуск выключен по умолчанию, чтобы случайно не потратить API-бюджет.
RUN_AGENT = False

# Пример настройки:
# Лучше загрузить /content/.env с ROUTERAI_API_KEY, чем вписывать ключ в notebook.
# os.environ["ROUTERAI_API_KEY"] = "..."
# os.environ["ROUTERAI_BASE_URL"] = "https://routerai.ru/api/v1"
# os.environ["AGENT_MODEL"] = "deepseek/deepseek-v4-pro"
# os.environ["SEARCH_PROVIDER"] = "none"

agent_results_path = OUTPUT_DIR / "agent_validation_results.jsonl"
agent_eval_results_path = OUTPUT_DIR / "agent_eval_results.jsonl"
agent_submission_path = OUTPUT_DIR / "submission_agent.csv"

if RUN_AGENT:
    cmd = [
        "python",
        "/content/agent_runner.py",
        "--env-file",
        "/content/.env",
        "--cases",
        str(OUTPUT_DIR / "agent_validation_cases.jsonl"),
        "--out",
        str(agent_results_path),
        "--limit",
        "50",
        "--search-provider",
        os.getenv("SEARCH_PROVIDER", "none"),
        "--fewshot-path",
        str(OUTPUT_DIR / "agent_fewshot_train_examples.jsonl"),
        "--fewshot-k",
        "3",
        "--evaluate",
    ]
    subprocess.run(cmd, check=True)
else:
    print("RUN_AGENT=False. Set RUN_AGENT=True after configuring API keys.")
    print("Cases:", OUTPUT_DIR / "agent_validation_cases.jsonl")
    print("Results will be saved to:", agent_results_path)

# Eval запуск выключен отдельно: он может стоить денег.
# Включать только после выбора дешевого провайдера/модели.
RUN_AGENT_ON_EVAL = False

if RUN_AGENT_ON_EVAL:
    cmd = [
        "python",
        "/content/agent_runner.py",
        "--env-file",
        "/content/.env",
        "--cases",
        str(OUTPUT_DIR / "agent_eval_cases.jsonl"),
        "--out",
        str(agent_eval_results_path),
        "--search-provider",
        os.getenv("SEARCH_PROVIDER", "none"),
        "--fewshot-path",
        str(OUTPUT_DIR / "agent_fewshot_train_examples.jsonl"),
        "--fewshot-k",
        "3",
        "--submission-csv",
        str(agent_submission_path),
    ]
    subprocess.run(cmd, check=True)
else:
    print("RUN_AGENT_ON_EVAL=False. Agent eval cases:", OUTPUT_DIR / "agent_eval_cases.jsonl")
    print("Agent submission will be saved to:", agent_submission_path)

## 4. Предсказания на eval

Эта секция сохраняет предсказания. Не используем `relevance_new` для выбора параметров.

Если усиленный ensemble выше отработал успешно, основной кандидат для отправки - `submission_ensemble.csv`. Одиночный `submission_transformer.csv` ниже остается запасным вариантом.

Для финального прогона можно поставить `TRAIN_FINAL_TRANSFORMER_ON_FULL_TRAIN = True`, чтобы переобучить трансформер на всём train после выбора настроек на validation.

In [ ]:
# TF-IDF eval predictions: train on all train data.
tfidf_full = make_tfidf_pipeline()
tfidf_full.fit(train_data["text"], train_data["label_id"])
tfidf_eval_pred_ids = tfidf_full.predict(eval_data["text"])
tfidf_eval_pred_labels = decode_label_ids(tfidf_eval_pred_ids)

tfidf_eval_out = eval_data.copy()
tfidf_eval_out["predicted_relevance"] = tfidf_eval_pred_labels
tfidf_eval_out.to_csv(
    OUTPUT_DIR / "tfidf_eval_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

display(tfidf_eval_out["predicted_relevance"].value_counts().sort_index())

In [ ]:
TRAIN_FINAL_TRANSFORMER_ON_FULL_TRAIN = False

prediction_trainer = trainer
prediction_tokenizer = tokenizer

if TRAIN_FINAL_TRANSFORMER_ON_FULL_TRAIN:
    full_hf_train = make_hf_dataset(train_data)
    tokenized_full_train = full_hf_train.map(tokenize_batch, batched=True, remove_columns=["text"])

    final_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABEL_VALUES),
        id2label={i: str(ID_TO_LABEL[i]) for i in LABEL_IDS},
        label2id={str(label): label_id for label, label_id in LABEL_TO_ID.items()},
    )

    final_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR / "final_transformer_checkpoints"),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=0.01,
        eval_strategy="no",
        save_strategy="epoch",
        save_total_limit=1,
        logging_steps=50,
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=SEED,
    )

    prediction_trainer = Trainer(
        model=final_model,
        args=final_args,
        train_dataset=tokenized_full_train,
        data_collator=data_collator,
        **trainer_tokenizer_kwargs(),
    )
    prediction_trainer.train()
    prediction_trainer.save_model(str(OUTPUT_DIR / "final_transformer_model"))

eval_hf = Dataset.from_pandas(eval_data[["text"]], preserve_index=False)
tokenized_eval = eval_hf.map(tokenize_batch, batched=True, remove_columns=["text"])
transformer_eval_output = prediction_trainer.predict(tokenized_eval)
if "BEST_LOGIT_BIAS" in globals():
    transformer_eval_pred_ids = predict_with_bias(transformer_eval_output.predictions, BEST_LOGIT_BIAS)
else:
    transformer_eval_pred_ids = np.argmax(transformer_eval_output.predictions, axis=-1)
transformer_eval_pred_labels = decode_label_ids(transformer_eval_pred_ids)

transformer_eval_out = eval_data.copy()
transformer_eval_out["predicted_relevance"] = transformer_eval_pred_labels
transformer_eval_out.to_csv(
    OUTPUT_DIR / "transformer_eval_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

submission = pd.DataFrame(
    {
        "permalink": eval_data.get("permalink"),
        "Text": eval_data.get("Text"),
        "relevance": transformer_eval_pred_labels,
    }
)
submission.to_csv(
    OUTPUT_DIR / "submission_transformer.csv",
    index=False,
    encoding="utf-8-sig",
)

display(transformer_eval_out["predicted_relevance"].value_counts().sort_index())
print("Saved:", OUTPUT_DIR / "transformer_eval_predictions.csv")
print("Saved:", OUTPUT_DIR / "submission_transformer.csv")

## 5. Архив результатов

Для сдачи и отчёта обычно достаточно лёгкого архива `final_artifacts.zip`: он не включает гигабайтные веса моделей, но содержит submission, validation metrics/predictions и конфиг ансамбля.

Полный архив всего `/content/outputs` можно сделать отдельно, если нужно сохранить обученные чекпойнты.

In [ ]:
import shutil

FINAL_ARTIFACTS_DIR = Path("/content/final_artifacts")
shutil.rmtree(FINAL_ARTIFACTS_DIR, ignore_errors=True)
FINAL_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

artifact_paths = [
    OUTPUT_DIR / "submission_ensemble.csv",
    OUTPUT_DIR / "ensemble_eval_predictions.csv",
    OUTPUT_DIR / "ensemble_valid_predictions.csv",
    OUTPUT_DIR / "ensemble_errors_sample.csv",
    OUTPUT_DIR / "submission_transformer.csv",
    OUTPUT_DIR / "transformer_eval_predictions.csv",
    OUTPUT_DIR / "transformer_calibrated_valid_predictions.csv",
    OUTPUT_DIR / "transformer_logit_bias.json",
    OUTPUT_DIR / "valid_split_preview.csv",
    OUTPUT_DIR / "strong_ensemble" / "model_scores.csv",
    OUTPUT_DIR / "strong_ensemble" / "ensemble_config.json",
]

for src in artifact_paths:
    if src.exists():
        rel = src.relative_to(OUTPUT_DIR)
        dst = FINAL_ARTIFACTS_DIR / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print("copied:", src)
    else:
        print("skip missing:", src)

shutil.make_archive("/content/final_artifacts", "zip", FINAL_ARTIFACTS_DIR)
print("Light archive:", "/content/final_artifacts.zip")

# Если нужно сохранить вообще всё, включая большие чекпойнты моделей, включи:
CREATE_FULL_OUTPUTS_ARCHIVE = False
if CREATE_FULL_OUTPUTS_ARCHIVE:
    shutil.make_archive("/content/yandex_maps_outputs", "zip", OUTPUT_DIR)
    print("Full archive:", "/content/yandex_maps_outputs.zip")

## 6. Что делать дальше

1. Сравнить TF-IDF и transformer на validation.
2. По `*_errors_sample.csv` понять, какие типы ошибок повторяются.
3. Если `submission_ensemble.csv` создан и validation accuracy выше одиночной модели, использовать его как основной submit.
4. Агентский слой оставить как отдельный эксперимент: запускать только на спорных случаях и сравнивать с transformer/ensemble на validation.